In [37]:
#Imports
import pandas as pd
import os.path
import random

#All Function definitions

def generateIDs(keyname, numberOfQuestions):
    '''generates a unique ID for each question using the beginning characters of each question'''
    goodIDs=[]
    with open(keyname) as file:  
        data = file.read()
    for n in range(1,numberOfQuestions+1):
            #print("processing: "+str(n))
            locs=[]
            occursat=find_all(data, str(n)+')')
            for ii in occursat:
                if data[ii-1]=='\r' or data[ii-1]=='\n'  and data[ii+3]!="_":# check for both /r and /n
                    locs.append(ii)
                    #print(locs)
            z=lambda x: str('0')+str(x) if x<10 else str(x)
            theID=z(n)+"_"+(data[locs[0]+3:locs[0]+25]).strip()
            goodIDs.append(theID)
    return goodIDs

def unduplicate(dup_list):
    '''finds duplicate entries in a list and adds characters to duplicated entries to make them unique.
    works in a roundabout way by converting to a dataframe first , and then converting back to a list in the end'''
    df = pd.DataFrame(dup_list)
    dups=df[df.duplicated(keep='first')]
    if not dups.empty:
        dup_indices=list(dups.index)
        for i in dup_indices:
            df.iat[i,0]=df.iat[i,0]+str(i)
            print("Unduplicating: "+ df.iat[i,0])
    undup_list=list(df.values.flatten())
    return undup_list

def createQuestionIDs(numberOfQuestions):
    '''this will generateIDs unique question IDs for each question from the raw keys'''
    allIDs=[]
    for n in range(0,6):
        filename="key"+str(n)+".txt"
        if (os.path.isfile(filename)): 
            qid=generateIDs(filename,numberOfQuestions)
            allIDs.append(qid)
    allIDs[0]=unduplicate(allIDs[0])
 #The ID's generated are slightly different in each version, so set them all equal to the ID's in version0           
    for n in range(1,numberOfVersions):
        for m in range(0,numberOfQuestions):
            sub=allIDs[0][m][4:20]
            matching = [s for s in allIDs[n] if sub in s]
            found_in=allIDs[n].index(matching[0])
            if allIDs[n][found_in]!=allIDs[0][m]:
                #print(QIDs[n][found_in]+"   changed to    "+QIDs[0][m])
                allIDs[n][found_in]=allIDs[0][m]
    
    return allIDs

def find_all(a_str, sub):
    '''finds all instances of a substring in a string'''
    start = 0
    result=[]
    while True:
        start = a_str.find(sub, start)
        if start == -1: return result
        result.append(start)
        start += len(sub) # use start += 1 to find overlapping matches
    return result

def cleanKey(key_name):
    '''retrieves just the "key" part of exported test. 
    Just reads and returns the key if the questions have already been deleted 
    writes to a new file with name cleankey'''
    if(os.path.isfile(key_name)):
        #print("Scrubbing: "+key_name+" which is :" + str(type(key_name)))
        with open(key_name) as file:  
            data = file.read()
            p=data.rfind('_____')
            if p!=-1: 
                data=data[p+5:]#delete all the junk before "_______"
                data=data[data.find('1)'):]#keep all the stuff after the 1)
    endline=['\rID:', '\nID:']
    for pp in endline:
        if data.find(pp)!=-1:# remove the question ID to a separate column if necessary
            data = data.replace(pp,'\t')
    with open('cleaned_'+key_name, 'w') as file:
        file.write(data)
        
def getAllKeys():
    '''read all files named key0-key4, make them into strings and and return a dictionary
    containing a list of keys and also the number of questions'''
    keylist=[]

    for n in range(0,6):
        filename="key"+str(n)+".txt"
        #print("processing: "+filename)
        if os.path.isfile(filename): 
            cleanKey(filename)#always create a fresh cleankey, to avoid using an outdated one
            keys=pd.read_table("cleaned_"+filename, header=None,usecols=[0])
            keylist.append(makekey(keys))
    numberOfQuestions=keys.shape[0]
    getAllKeysOutput={"keylist":keylist,"numberOfQuestions":numberOfQuestions}
    return getAllKeysOutput


def makekey(key_from_test):
    '''key_from_test is a dataframe formed from reading the answerkey file. This function converts it into a text string with numbers 0-5 representing A-E'''
    #keydictionary={"A":"0","B":"1","C":"2","D":"3","E":"4"}
    thekey=""
    for i in range(0,key_from_test.shape[0]):
        #print("At i="+str(i))
        #get the last letter, i.e. the "C" from "1)C" 
        thekey+=keydictionary[key_from_test.iat[i,0][-1:]]
    return thekey


def gradeWithKeylist(keylist, ans, numberOfQuestions, QIDs=[], points=[]):
    '''multiple versions - find the correct key as indicated on the last question on the exam '''
    assert(keylist!=[])
    whichKey=int(ans[-1:])
    key=keylist[whichKey]
    assert len(key)==len(ans)
    missed=""
    #invkeydictionary={0:"A",1:"B",2:"C",3:"D",4:"E"}
    rejalt=[1]*numberOfQuestions
    for n in range(0,len(key)-1):
        if key[n]!=ans[n]:
            #print("storing")
            missed+=str(n+1)+", "
            rejalt[n]=0
    if sum(rejalt)==numberOfQuestions:
        missed="ALL CORRECT"
    else:
        missed="v"+invkeydictionary[whichKey]+": "+missed[:len(missed)-2]
    

    if not points:
        points=[1]*numberOfQuestions
      
    points[-1]=0 #No points for the test version
    score=sum([i*j for i,j in zip(points,rejalt)])
      
    
    if QIDs:
        global analysis_df
        mydict = dict(zip(QIDs[whichKey],rejalt))
        sortedIDs=sorted(mydict.keys())
        sorted_rejalt=[mydict[k] for k in sortedIDs]
        sorted_rejalt.append(score)
        analysis_df.loc[len(analysis_df)] = sorted_rejalt 
        
        global allAnswers_df
        mydict = dict(zip(QIDs[whichKey],ans))  
        sortedIDs=sorted(mydict.keys())
        sorted_ans=[mydict[k] for k in sortedIDs]
        allAnswers_df.loc[len(allAnswers_df)] = sorted_ans
        
    return {'missed':missed, 'score':score}

def process_grades(keylist,data,numberOfQuestions, QIDs=[]):
    '''grades all exams using correct keys, writes questions missed and scores'''
    for NN in range(0, data.shape[0]):
        #print("NN: "+str(NN))
        check1=gradeWithKeylist(keylist, df.iat[NN,2],numberOfQuestions, QIDs)
        data.iat[NN,3]=check1['missed']
        data.iat[NN,4]=check1['score']
    return data

 ####################### SUGGESTED USAGE#################################
# outs = getAllKeys()
# keylist=outs["keylist"]
# numberOfQuestions=outs["numberOfQuestions"] 
# numberOfVersions=len(keylist)

# QIDs=createQuestionIDs(numberOfQuestions)
# analysis_df  = pd.DataFrame(columns = QIDs[0])

In [38]:
#Manually convert the dat file into an excel file using excel. Only extract the serial number,
#name and Answers, and make sure Answers is text.
# Do the following by hand:
#check serial numbers (using remove duplicates)
#manually fill in missing information 
rawdatafilename='Allsmall'
xls_file = pd.ExcelFile(rawdatafilename+'.xlsx', dtype=str)
df = xls_file.parse('Sheet1', header=None, parse_cols=2,names = ["Srl No", "Name", "Answers"])
#parse_cols makes sure that only cols 0,1 and 2 are extracted
df["Missed"] = ""
df["Score"]=0

In [39]:
#here is where we run the code
keydictionary={"A":"0","B":"1","C":"2","D":"3","E":"4"}
invkeydictionary={0:"A",1:"B",2:"C",3:"D",4:"E"}
outs = getAllKeys()
keylist=outs["keylist"]
numberOfQuestions=outs["numberOfQuestions"] 
numberOfVersions=len(keylist)
numberOfStudents=df.shape[0]


QIDs=createQuestionIDs(numberOfQuestions)
headings=QIDs[0]
allAnswers_df = pd.DataFrame(columns = headings)# stores all student answers for each question, questions are the columns
headings.append("score")
analysis_df  = pd.DataFrame(columns = headings)# stores whether answer was correct, questions are the columns, last column is the score
df=process_grades(keylist,df,numberOfQuestions, QIDs)

In [ ]:
#Grades and displays the N'th entry in the list using a keylist
N=random.randint(1,df.shape[0]-1)
check1=gradeWithKeylist(keylist, df.iat[N,2],numberOfQuestions, QIDs)
print("Row "+ str(N)+": "+df.iat[N,1]+", "+str(df.iat[N,0])+". Missed "+str(check1['missed'])+ ". Scored "+str(check1['score'])+"/"+str(numberOfQuestions-1))

In [ ]:
# writing to csv
df.to_csv(rawdatafilename+'_processed.csv')
analysis_df.to_csv(rawdatafilename+'_analysis.csv')
#### Don't opoen this file directly in excel, since the "Answers" column gets messed up. Import it using the import wizard. 

In [40]:
df

,Srl No,Name,Answers,Missed,Score
0,1001,TRAN KHANH N,10310004130104202442213422,"vC: 14, 16",23
1,1002,HAO SHIRA,10310034130102212442213422,vC: 7,24
2,1003,RUTAYISIRE DIANDRY A,10303032430441314203221120,"vA: 7, 13, 19, 23",21
3,1004,RUIZ TYLER J,10303022410421214211220120,"vA: 10, 15, 20, 23",21
4,1005,JIANG JASMINE,10303022430421314203223120,vA: 19,24
5,1006,SHIELDS TIFFANY,20333021430421314213223120,"vA: 1, 4, 8",22
6,1007,SHIN DANIEL,10410034131102212442210422,"vC: 3, 7, 11, 23",21


In [41]:
allAnswers_df

,01_A mass M is attached t,02_A dumbbell-shaped obje,03_A 1.2-kg spring-activa,04_During a collision wit,"05_A meteoroid, heading s",06_A boy on a bicycle app,07_An unusual spring is p,08_A heavy stone of mass,09_If right ray(B) = -2ha,10_A uniform sign is sup,...,17_A particle moving in,18_Two loudspeakers plac,19_A proton (a positivel,20_A uniform solid cylin,21_Observer A is 3.0 m f,22_The sound level at 1.,23_For the (conservative,24_You are generating tr,"25_In the figure, a bloc",26_What version is this
0,0,4,1,3,2,1,4,4,4,0,...,0,0,2,4,0,1,3,2,1,2
1,0,4,1,3,2,1,4,4,4,0,...,1,0,2,2,0,1,3,2,1,2
2,1,0,3,0,3,0,3,2,4,3,...,4,2,0,3,2,2,1,1,2,0
3,1,0,3,0,3,0,2,2,4,1,...,4,2,1,1,2,2,0,1,2,0
4,1,0,3,0,3,0,2,2,4,3,...,4,2,0,3,2,2,3,1,2,0
5,2,0,3,3,3,0,2,1,4,3,...,4,2,1,3,2,2,3,1,2,0
6,0,4,1,4,2,1,4,4,4,0,...,1,0,2,2,0,1,0,2,1,2


In [42]:
analysis_df

,01_A mass M is attached t,02_A dumbbell-shaped obje,03_A 1.2-kg spring-activa,04_During a collision wit,"05_A meteoroid, heading s",06_A boy on a bicycle app,07_An unusual spring is p,08_A heavy stone of mass,09_If right ray(B) = -2ha,10_A uniform sign is sup,...,18_Two loudspeakers plac,19_A proton (a positivel,20_A uniform solid cylin,21_Observer A is 3.0 m f,22_The sound level at 1.,23_For the (conservative,24_You are generating tr,"25_In the figure, a bloc",26_What version is this,score
0,1,1,1,1,1,1,1,1,1,1,...,1,1,0,1,1,1,1,1,1,23
1,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,24
2,1,1,1,1,1,1,0,1,1,1,...,1,0,1,1,1,0,1,1,1,21
3,1,1,1,1,1,1,1,1,1,0,...,1,1,0,1,1,0,1,1,1,21
4,1,1,1,1,1,1,1,1,1,1,...,1,0,1,1,1,1,1,1,1,24
5,0,1,1,0,1,1,1,0,1,1,...,1,1,1,1,1,1,1,1,1,22
6,1,1,1,0,1,1,1,1,1,1,...,1,1,1,1,1,0,1,1,1,21


In [ ]:
u=analysis_df.iloc[:,0]
count=0
for n in range(0,u.shape[0]):
    if u[n]==0:
        count+=1
print(count)
        

In [ ]:
mydf = pd.DataFrame(columns = [QIDs[0],"score"])
    


In [ ]:
type(QIDs[0])

In [ ]:
QIDs[0].append('score')

In [ ]:
QIDs[0]